# TP3. Reconstrucción 3D y estimación de pose

Visión por Computadora - Robótica Móvil

**Cómo usarlo:** Dejar este notebook en la raíz del repo (`TP3/`), completar la celda de configuración y ejecutar todo con *Run All*. En caso de que haya salidas mostradas, hacer un Restart Kernel para borrar cualquier estado viejo.

Las figuras obtenidas se guardan en `figures/`.

> El nombre de las imagenes stereo originales es el mismo, tanto para cámara izquierda como derecha. Se identidican de acuerda a la carpeta donde están.
> Esto es así porque en los dataset las imagenes se guardan con el nombre de acuerdo al timestamp de cuando fueron capturadas.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import yaml
import pandas as pd


ROOT = Path.cwd() # La notebook va en la raíz del repo (TP3)
CALIB_DIR = ROOT / "results/calibration" # left.yaml y right.yaml obtenidos con la calibración del Ej 1
SEQ_DIR = ROOT / "data/stereo_example" # Carpeta que contiene los directorios cam0 y cam1
FRAME = "1403636579763555584.png" # Nombre de las imagenes ()
FIG_DIR = ROOT / "figures" # Acá se guardan las figuras obtenidas a lo largo de los ejercicio

FIG_DIR.mkdir(exist_ok=True)
LEFT_PATH = SEQ_DIR / "cam0" / FRAME
RIGHT_PATH = SEQ_DIR / "cam1" / FRAME

#print(LEFT_PATH)
#print(RIGHT_PATH)

In [ ]:
img1 = cv2.imread(LEFT_PATH)
img2 = cv2.imread(RIGHT_PATH)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(img1, cmap='gray')
axes[0].axis('off')
axes[0].set_title("Cámara Izquierda")

axes[1].imshow(img2, cmap='gray')
axes[1].axis('off')
axes[1].set_title("Cámara derecha")

plt.tight_layout()
plt.show()

In [ ]:
def show(img, title="", size=(12, 5)):
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()


def save(img, name):
    cv2.imwrite(str(FIG_DIR / name), img)

# Ej. 1. Calibración de la cámara estéreo

Calibración hecha con `ros2 run camera_calibration cameracalibrator` sobre `cam_checkerboard` (tablero 7×6). Los yaml que guarda el calibrador traen `K` (intrínseca), `D` (distorsión), y la rectificación (`R`, `P`) de cada cámara. La pose entre cámaras y el error de reproyección solo salieron por terminal; están copiados más abajo.

## Salida en la terminal luego de la calibración

```shell
Stereo RMS re-projection error: 0.2659409146589982

Left:
D = [-0.28454587860679176, 0.07562820138166555, 0.0003791677575551562, 0.00012090717268091055, 0.0]
K = [461.67318458071037, 0.0, 368.0679817907219, 0.0, 460.4108640633701, 247.63174634147535, 0.0, 0.0, 1.0]
R = [0.9999855127648051, -0.0012956153536859353, 0.005224523075363776, 0.001260931031323702, 0.9999771825905618, 0.006636591834408091, -0.005233002335558448, -0.006629907925271702, 0.9999643293675321]
P = [439.2536534174921, 0.0, 371.33053970336914, 0.0, 0.0, 439.2536534174921, 256.0329170227051, 0.0, 0.0, 0.0, 1.0, 0.0]

Right:
D = [-0.2866006602364961, 0.07774236863472772, -0.0003191969270104431, 0.0003244000114631708, 0.0]
K = [460.97751449612394, 0.0, 375.6607261513001, 0.0, 460.0253197978464, 254.888853512581, 0.0, 0.0, 1.0]
R = [0.9999727666707475, -0.0036153003111019976, -0.006433935072045855, 0.003572542427770412, 0.9999715413022566, -0.006644811975601251, 0.006457774961434948, 0.006621645509226493, 0.9999572245617796]
P = [439.2536534174921, 0.0, 371.33053970336914, -48.132983792612535, 0.0, 439.2536534174921, 256.0329170227051, 0.0, 0.0, 0.0, 1.0, 0.0]
self.T = [-0.10957603333914648, 0.0003961610562057416, 0.0007050242841150852]
self.R = [0.9999289910082404, 0.0022340663884456383, 0.01170563490863621, -0.002389003874944096, 0.9999095076807233, 0.013238924056699056, -0.011674999003330461, -0.013265948781205223, 0.999843842307991]
None
# oST version 5.0 parameters


[image]

width
752

height
480

[narrow_stereo/left]

camera matrix
461.673185 0.000000 368.067982
0.000000 460.410864 247.631746
0.000000 0.000000 1.000000

distortion
-0.284546 0.075628 0.000379 0.000121 0.000000

rectification
0.999986 -0.001296 0.005225
0.001261 0.999977 0.006637
-0.005233 -0.006630 0.999964

projection
439.253653 0.000000 371.330540 0.000000
0.000000 439.253653 256.032917 0.000000
0.000000 0.000000 1.000000 0.000000
# oST version 5.0 parameters


[image]

width
752

height
480

[narrow_stereo/right]

camera matrix
460.977514 0.000000 375.660726
0.000000 460.025320 254.888854
0.000000 0.000000 1.000000

distortion
-0.286601 0.077742 -0.000319 0.000324 0.000000

rectification
0.999973 -0.003615 -0.006434
0.003573 0.999972 -0.006645
0.006458 0.006622 0.999957

projection
439.253653 0.000000 371.330540 -48.132984
0.000000 439.253653 256.032917 0.000000
0.000000 0.000000 1.000000 0.000000

('Wrote calibration data to', '/tmp/calibrationdata.tar.gz')
('Wrote calibration data to', '/tmp/calibrationdata.tar.gz')
```

In [ ]:
def load_camera_info(path):
    with open(path) as f:
        d = yaml.safe_load(f)
    K = np.array(d["camera_matrix"]["data"], dtype=np.float64).reshape(3, 3)
    D = np.array(d["distortion_coefficients"]["data"], dtype=np.float64).reshape(5, 1)
    R = np.array(d["rectification_matrix"]["data"], dtype=np.float64).reshape(3, 3)
    P = np.array(d["projection_matrix"]["data"], dtype=np.float64).reshape(3, 4)
    size = (d["image_width"], d["image_height"])
    return K, D, R, P, size


Kl, Dl, Rl, Pl, size = load_camera_info(CALIB_DIR / "left.yaml")
Kr, Dr, Rr, Pr, _ = load_camera_info(CALIB_DIR / "right.yaml")


T_stereo = np.array([-0.10957603333914648, 0.0003961610562057416, 0.0007050242841150852])

R_stereo = np.array([[0.9999289910082404, 0.0022340663884456383, 0.01170563490863621],
                     [-0.002389003874944096, 0.9999095076807233, 0.013238924056699056],
                     [-0.011674999003330461, -0.013265948781205223, 0.999843842307991]])
RMS = 0.2659409146589982

#print(Dl)
#print(Dr)

# Ej. 2a. Rectificar imágenes

Se usan directamente `R` y `P` de los yaml en `initUndistortRectifyMap`: desdistorsionan, rotan y reproyectan con la `K'` común en un solo `remap`.

In [ ]:
# Cargamos las imgs de prueba
left_raw = cv2.imread(str(LEFT_PATH), cv2.IMREAD_GRAYSCALE)
right_raw = cv2.imread(str(RIGHT_PATH), cv2.IMREAD_GRAYSCALE)


def rectify(img, K, D, R, P, size):
    # Calculamos donde mandar los pixeles para que la imagen se rectifique
    map1, map2 = cv2.initUndistortRectifyMap(K, D, R, P[:, :3], size, cv2.CV_32FC1)
    # Aplicamos la rectificación
    return cv2.remap(img, map1, map2, cv2.INTER_LINEAR)


def with_lines(left, right, step=40):
    out = cv2.cvtColor(np.hstack([left, right]), cv2.COLOR_GRAY2BGR)
    for v in range(step, out.shape[0], step):
        cv2.line(out, (0, v), (out.shape[1] - 1, v), (0, 255, 0), 2)
    return out

left_rect = rectify(left_raw, Kl, Dl, Rl, Pl, size)
right_rect = rectify(right_raw, Kr, Dr, Rr, Pr, size)

cmp_raw = with_lines(left_raw, right_raw)
cmp_rect = with_lines(left_rect, right_rect)
show(cmp_raw, "Sin rectificar", (14, 4))
show(cmp_rect, "Rectificado", (14, 4))
save(cmp_raw, "2a_raw.png")
save(cmp_rect, "2a_rect.png")

## Ej. 2b. Keypoints y descriptores

ORB (Oriented FAST and Rotated BRIEF) sobre las imágenes rectificadas.

In [ ]:
# Configuramos ORB
orb = cv2.ORB_create(nfeatures=5000)

# Obtenemos de una pasada la ubiicacion de los keypoints (kp_l | kp_r)
# y los descriptores (desc_l | desc_r)
kp_l, desc_l = orb.detectAndCompute(left_rect, None)
kp_r, desc_r = orb.detectAndCompute(right_rect, None)

print(f"Keypoints: izquierda={len(kp_l)}, derecha={len(kp_r)}")
print(f"Info descriptores: {desc_l.shape}, {desc_l.dtype} ({desc_l.shape[1] * 8} bits)")


# Obtenemos una matriz que representa el dibujo de los keypoints
vis_simple_l = cv2.drawKeypoints(
    image=left_rect, 
    keypoints=kp_l, 
    outImage=None, 
    color=(0, 255, 0)
)

vis_simple_r = cv2.drawKeypoints(
    image=right_rect, 
    keypoints=kp_r, 
    outImage=None, 
    color=(0, 255, 0)
)

# Ponemos una imagen (matriz) al lado de la otra
vis_kp = np.hstack([vis_simple_l, vis_simple_r])

show(vis_kp, "Keypoints ORB", (14, 4))
save(vis_kp, "2b_keypoints.png")

## Ej.2c. Matching

`BFMatcher` con distancia de Hamming. Se muestran todos los matches y los de distancia < 20.

In [ ]:
MAX_DIST = 25

matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
matches = matcher.match(desc_l, desc_r)

good = [m for m in matches if m.distance < MAX_DIST]
print(f"Todos los matches: {len(matches)} | con distancia menor a {MAX_DIST}: {len(good)}")

flags = cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS
vis_all = cv2.drawMatches(left_rect, kp_l, right_rect, kp_r, matches, None, matchColor=(0, 255, 0), flags=flags)
vis_good = cv2.drawMatches(left_rect, kp_l, right_rect, kp_r, good, None, matchColor=(0, 255, 0), flags=flags)
show(vis_all, f"Todos los matches ({len(matches)})", (14, 4))
show(vis_good, f"Matches con distancia < {MAX_DIST} ({len(good)})", (14, 4))
save(vis_all, "2cmatches_all.png")
save(vis_good, f"2cmatches_dist{MAX_DIST}.png")

## Ej. 2d. Triangulación de puntos 3D

**Nota sobre la función.** El enunciado pide `cv::sfm::triangulatePoints`, que pertenece al módulo `sfm` de opencv_contrib. Ese módulo parece no estár en los paquetes de pip (y no lo pude instalar, ja), así que se usó `cv2.triangulatePoints`

**Visualización.** El enunciado sugiere publicar un `PointCloud2` en RViz. Acá se muestra la nube con matplotlib, con el color del píxel de la imagen izquierda en cada punto. Además se dibuja cada punto sobre la imagen coloreado por profundidad.

In [ ]:
# Píxeles de los matches buenos (2c), en formato 2xN que nos pide cv2.triangulatePoints
pts_l = np.array([kp_l[m.queryIdx].pt for m in good], dtype=np.float64).T
pts_r = np.array([kp_r[m.trainIdx].pt for m in good], dtype=np.float64).T

# Le pasamos a la función trianguladora las matrices de proyección y los puntos obtenidos anteiormente
X_h = cv2.triangulatePoints(Pl, Pr, pts_l, pts_r)
# Como X_h es homogenea, lo pasamos a coordenada normal (cartesianas)
X = (X_h[:3] / X_h[3]).T

# Mostramos solo puntos delante de la cámara y a menos de 20 m (para que un punto muy lejano no me aplaste el eje)
ok = (X[:, 2] > 0) & (X[:, 2] < 20)
X, pts_ok = X[ok], pts_l[:, ok] # El enunciado pide cámara izquierda

# Figura 1- puntos sobre la imagen izquierda
vis_3d = cv2.cvtColor(left_rect, cv2.COLOR_GRAY2BGR)


z_norm = np.clip(X[:, 2] / 10 * 255, 0, 255).astype(np.uint8)
colors_z = cv2.applyColorMap(z_norm, cv2.COLORMAP_JET).squeeze()

for (u, v), color in zip(pts_ok.T, colors_z):
    cv2.circle(vis_3d, (int(u), int(v)), 3, color.tolist(), -1)

show(vis_3d, "Puntos triangulados (azul: más cerca, rojo: más lejos)")
save(vis_3d, "2d_depth.png")


# Figura 2- Nube de puntos 3D
u, v = np.round(pts_ok).astype(int)

pixel_intensity = left_rect[v, u] 

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")

ax.scatter(X[:, 0], X[:, 2], -X[:, 1], c=pixel_intensity, cmap="gray", s=12)

for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.set_pane_color((0, 0, 0, 1))
ax.set_xlabel("X[m]"); ax.set_ylabel("Z[m] (adelante)"); ax.set_zlabel("-Y[m] (arriba)")
ax.view_init(elev=20, azim=-70)

fig.savefig(FIG_DIR / "2d_pointcloud.png", dpi=150)
plt.show()


## Ej. 2e. Filtrado de correspondencias espúreas (RANSAC y matriz fundamental)

Los matches tienen outliers: pares de keypoints que no son el mismo punto 3D.
RANSAC loopea sobre lo siguiente:
1. sortear el mínimo de matches necesario para ajustar un modelo
2. calcular el modelo con esa muestra
3. contar cuántos matches lo cumplen (inliers).

Se queda con el modelo con más inliers

In [ ]:

MAX_DIST_E = 50 # Cambiamos el filtro para esta prueba
cand = [m for m in matches if m.distance < MAX_DIST_E]

pts_l = np.float32([kp_l[m.queryIdx].pt for m in cand])
pts_r = np.float32([kp_r[m.trainIdx].pt for m in cand])

# Matriz fundamental y máscara de matches inliers y outliers
F, mask = cv2.findFundamentalMat(
    pts_l, pts_r, cv2.FM_RANSAC, 1.0, 0.999
)
inl = mask.ravel().astype(bool)

print(f"Matches: {len(cand)} | Inliers: {inl.sum()} | Outliers: {(~inl).sum()}")

# Mostramos matches. Verdes = inliers, rojos = outliers
flags = cv2.DRAW_MATCHES_FLAGS_NOT_DRAW_SINGLE_POINTS

vis = cv2.drawMatches(
    left_rect, kp_l, right_rect, kp_r,
    [m for m, ok in zip(cand, inl) if not ok],
    None, matchColor=(0, 0, 255), flags=flags
)
vis = cv2.drawMatches(
    left_rect, kp_l, right_rect, kp_r,
    [m for m, ok in zip(cand, inl) if ok],
    vis, matchColor=(0, 255, 0),
    flags=flags | cv2.DRAW_MATCHES_FLAGS_DRAW_OVER_OUTIMG
)

show(vis, f"RANSAC: {inl.sum()} inliers, {(~inl).sum()} outliers", (14, 4))
save(vis, "2e_matches_ransac.png")


# Triangulación de puntos de nuevo
def triangular(pl, pr):
    X = cv2.triangulatePoints(Pl, Pr, pl.T.astype(np.float64), pr.T.astype(np.float64))
    return (X[:3] / X[3]).T


# Color de cada punto según la imagen original izquierda
x = np.round(pts_l[:, 0]).astype(int)
y = np.round(pts_l[:, 1]).astype(int)
colors = left_rect[y, x] / 255.0

def draw_pointcloud(ax, P, colors, title):
    # Mostramos solo puntos delante de la cámara y a menos de 20 m (para que un punto muy lejano no me aplaste el eje)
    ok = (P[:, 2] > 0) & (P[:, 2] < 20)

    ax.scatter(P[ok, 0], P[ok, 2], c=colors[ok], s=8)
    ax.plot(0, 0, "r^", ms=10)
    ax.set_title(f"{title}: {ok.sum()} puntos")
    ax.set_xlabel("X [m]")
    ax.set_facecolor("black")
    ax.grid(alpha=0.3)


# Antes y después de RANSAC
P_before = triangular(pts_l, pts_r)
P_after = triangular(pts_l[inl], pts_r[inl])

fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

draw_pointcloud(axs[0], P_before, colors, "Antes de RANSAC")
draw_pointcloud(axs[1], P_after, colors[inl], "Después de RANSAC")

axs[0].set_ylabel("Z[m] (adelante)")
axs[0].set_xlim(-8, 10)
fig.tight_layout()
fig.savefig(FIG_DIR / "2e_pointcloud.png", dpi=150)
plt.show()


# Obtener líneas epipolares en la imagen derecha
lines = cv2.computeCorrespondEpilines(pts_r[inl].reshape(-1, 1, 2), 2, F).reshape(-1, 3)

# Voy a elegir 15 puntos a azar
rng = np.random.default_rng(0)
idx = rng.choice(len(lines), size=15, replace=False)

vis_ep = cv2.cvtColor(left_rect, cv2.COLOR_GRAY2BGR)
h, w = vis_ep.shape[:2]

for i in idx:
    a, b, c = lines[i]
    u, v = pts_l[inl][i]

    color = tuple(map(int, rng.integers(80, 256, 3)))

    # Para dibujar una recta, no puedo usar los coeficinetes que me devuelve computeCorrespondEpilines
    y0 = int(round(-c / b))
    y1 = int(round(-(a * w + c) / b))

    cv2.line(vis_ep, (0, y0), (w, y1), color, 1)
    cv2.circle(vis_ep, (round(u), round(v)), 4, color, -1)

show(vis_ep, "Líneas epipolares en la imagen izquierda")
save(vis_ep, "2e_epilines.png")
